# LightGBM - S6E10 Predicting Airline Satisfaction

Experiments 2 and 3 (see `experiments.md` for the running log). Same data, same stratified 80/20 split,
same seed (42) as `random_forest.ipynb`, so every validation AUC in the log is comparable.

Why gradient boosting next: on tabular data it usually beats random forests, because each new tree corrects
the previous ones' errors instead of voting independently. LightGBM also handles categorical features and
missing values natively - no one-hot encoding, no imputation, no scaling.

Reference points from the log: logistic 0.92046, random forest 0.95351 (same split).


In [1]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
RF_VAL_AUC = 0.95351  # experiment 1, same split


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
categorical_columns = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
for col in categorical_columns:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

features = train.drop(columns=['id', 'satisfaction'])
test_features = test.drop(columns=['id'])

X_train, X_val, y_train, y_val = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,} rows, validation: {len(X_val):,} rows')


Train: 559,708 rows, validation: 139,927 rows


## Experiment 2: LightGBM, raw features

Mostly defaults, bounded budget: up to 2000 trees with early stopping on the validation split
(stops when 50 rounds add nothing). Early stopping uses the validation split to pick the tree count,
so treat this AUC as very mildly optimistic; k-fold is the fix when it matters.


In [3]:
model2 = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1,
)
# LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
# use whichever signature the installed version supports (no warning either way).
if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
    model2.fit(
        X_train, y_train,
        eval_X=X_val, eval_y=y_val,
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
else:
    model2.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
auc2 = roc_auc_score(y_val, model2.predict_proba(X_val)[:, 1])
print(f'Exp 2 (LightGBM raw) validation ROC AUC: {auc2:.5f}  (best_iteration={model2.best_iteration_})')
print(f'Delta vs random forest: {auc2 - RF_VAL_AUC:+.5f}')


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Exp 2 (LightGBM raw) validation ROC AUC: 0.95760  (best_iteration=418)
Delta vs random forest: +0.00409


## Experiment 3: LightGBM + three engineered features

One controlled change on top of Exp 2, guided by the RF importance table and domain sense:

- `total_delay` = departure + arrival delay (overall disruption)
- `delay_diff` = arrival - departure delay (did the flight recover time?)
- `service_mean` = mean of the 13 service-rating columns (overall service impression)

If Exp 3 does not beat Exp 2 on the same split, the features go in the log as a failed experiment - that is still a result.


In [4]:
RATING_COLUMNS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness',
]

def add_engineered(df):
    out = df.copy()
    out['total_delay'] = out['Departure Delay in Minutes'] + out['Arrival Delay in Minutes']
    out['delay_diff'] = out['Arrival Delay in Minutes'] - out['Departure Delay in Minutes']
    out['service_mean'] = out[RATING_COLUMNS].mean(axis=1)
    return out

X_train_e = add_engineered(X_train)
X_val_e = add_engineered(X_val)

model3 = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1,
)
if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
    model3.fit(
        X_train_e, y_train,
        eval_X=X_val_e, eval_y=y_val,
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
else:
    model3.fit(
        X_train_e, y_train,
        eval_set=[(X_val_e, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
auc3 = roc_auc_score(y_val, model3.predict_proba(X_val_e)[:, 1])
print(f'Exp 3 (LightGBM + engineered) validation ROC AUC: {auc3:.5f}  (best_iteration={model3.best_iteration_})')
print(f'Delta vs Exp 2: {auc3 - auc2:+.5f}')


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Exp 3 (LightGBM + engineered) validation ROC AUC: 0.95748  (best_iteration=290)
Delta vs Exp 2: -0.00012


## Feature importance (gain)

Gain-based importances from the better model - more reliable than the forest's impurity importances.


In [5]:
best_model = model3 if auc3 >= auc2 else model2
best_name = 'Exp 3' if auc3 >= auc2 else 'Exp 2'
importance_table = (
    pd.DataFrame({
        'feature': best_model.booster_.feature_name(),
        'gain': best_model.booster_.feature_importance(importance_type='gain'),
    })
    .sort_values('gain', ascending=False)
    .head(15)
)
print(f'Best: {best_name}')
importance_table


Best: Exp 2


,feature,gain
7,Online_boarding,2.807744e+06
2,Inflight_wifi_service,6.961709e+05
19,Type_of_Travel,6.731735e+05
18,Customer_Type,2.081343e+05
9,Inflight_entertainment,1.895125e+05
20,Class,1.675701e+05
13,Checkin_service,1.207508e+05
12,Baggage_handling,8.321449e+04
10,On-board_service,7.323892e+04
8,Seat_comfort,6.824017e+04


## Fit the winner on all data and write predictions

Refit the better configuration on every labeled row (using its early-stopped tree count) and write
`submission_lgbm.csv`. Uploading it to Kaggle is your action from your own account - it counts toward the daily limit.
Record the leaderboard score in `experiments.md` after submitting.


In [6]:
features_e = add_engineered(features)
test_features_e = add_engineered(test_features)

if auc3 >= auc2:
    final_X, final_test = features_e, test_features_e
    final_trees = model3.best_iteration_
else:
    final_X, final_test = features, test_features
    final_trees = model2.best_iteration_

final_model = lgb.LGBMClassifier(
    n_estimators=final_trees,
    learning_rate=0.05,
    num_leaves=63,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1,
)
final_model.fit(final_X, target)
test_probabilities = final_model.predict_proba(final_test)[:, 1]

submission = pd.read_csv(data_dir / 'sample_submission.csv')
submission['satisfaction'] = test_probabilities
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])

submission_path = competition_dir / 'submission_lgbm.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission):,} predictions to {submission_path} from {best_name} ({final_trees} trees).')
submission.head()


Saved 299,844 predictions to submission_lgbm.csv from Exp 2 (418 trees).


,id,satisfaction
0,699635,0.210542
1,699636,0.804964
2,699637,0.929608
3,699638,0.081729
4,699639,0.047498
